In [18]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import pandas as pd
import xarray as xr
from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils, isku_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'utils.isku_utils' from '/home/emily_zuetell/projects/poreallas/utils/isku_utils.py'>

In [12]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

# Define Forecast
EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]

In [13]:
RUN_MONTH = "2026-10"
N_LEADS = 6

_months = pd.period_range(pd.Period(RUN_MONTH, "M"), periods=N_LEADS, freq="M")
FC_MONTHS = [p.month for p in _months]
FC_PERIOD = slice(
    _months[0].start_time.strftime("%Y-%m-%d"),
    _months[-1].end_time.strftime("%Y-%m-%d"),
)

# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")


config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=False,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

In [14]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [15]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [16]:
# Compute impact: forecast - baseline
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config.dims}),
    config,
    ensemble=True,
)
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [17]:
# Aggregate Impact Regions to group_level
group_level = "ISO"  # IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
impact, merge_key, base_cols = analysis_utils.aggregate_impact(
    impact, config, group_level
)

In [ ]:
impact_sahel = impact.sel(ISO=["NGA", "SDN", "NER", "TCD"])
impact_seasia = impact.sel(ISO=["PHL", "VNM", "THA", "KHM"])
mean_sahel, se_sahel = analysis_utils.crossed_sem(
    impact_sahel.sum(dim=["month", "ISO"]), a="number", b="sample", include_resid=True
)
mean_seasia, se_seasia = analysis_utils.crossed_sem(
    impact_seasia.sum(dim=["month", "ISO"]), a="number", b="sample", include_resid=True
)
print(f"Sahel: {mean_sahel.values} +/- {1.96 * se_sahel.values}")
print(f"SE Asia: {mean_seasia.values} +/- {1.96 * se_seasia.values}")

In [21]:
analysis_utils.make_csv(effect, config, group_level="ISO")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_